In [237]:
words = open("names.txt","r").read().splitlines()

In [238]:
stoi = {ch: i for i,ch in enumerate(sorted(set(".".join(words))))}
itos = {i:ch for ch,i in stoi.items()}

In [239]:
import torch

In [240]:
context = []
context_len = 3
iy = []
for w in words:
    w = w+'.'
    cntxt = [0]*context_len
    for ch in w:
        context.append(cntxt)
        iy.append(stoi[ch])
        cntxt = cntxt[1:] + [stoi[ch]]
context = torch.tensor(context)
iy = torch.tensor(iy)
ix = context
context.shape, iy.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [241]:
emb_len = 15
cntx_len = 3



In [165]:
import torch

C = torch.rand((27,emb_len)).float()
W1 = torch.randn((emb_len*cntx_len, 100)).float()*(5/3)/torch.sqrt(torch.tensor(emb_len*cntx_len)).item()
b1 = torch.randn((100,)).float() * 0.01

W2 = torch.randn((100, 27)).float()*0.01
b2 = torch.randn((27,)).float()*0.01

running_mean = torch.zeros((1,100),dtype=torch.float32);
running_std = torch.ones((1,100),dtype=torch.float32);

gamma = torch.ones((1,100),dtype=torch.float32)
beta = torch.zeros((1,100),dtype=torch.float32)

parameters= [C,W1,W2,b1,b2,gamma,beta]

for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

7932

In [166]:
lli = torch.linspace(0,1,1000)
lossi = []

In [167]:
import torch.nn.functional as F

for i in range(20000):

    batch = torch.randint(0, iy.shape[0], (32,))
    emb = C[ix[batch]]
    h = emb.view(-1,emb_len*cntx_len) @ W1 + b1
    h = torch.tanh(h)

    mean_h = h.mean(dim = 0,keepdim=True)
    std_h = h.std(dim = 0, keepdim=True)

    h = gamma * (h - mean_h)/(std_h + 1e-5) + beta

    with torch.no_grad():
        running_mean = 0.99*running_mean + 0.01*mean_h
        running_std = 0.99*running_std + 0.01*std_h
    logits = h @ W2+ b2

    loss = F.cross_entropy(logits, iy[batch])

    for p in parameters:
        p.grad = None
        
    lr = 0.1 if i < 100000 else 0.01
    print(loss.item())
    loss.backward()
    for p in parameters:
        p.data -= lr*p.grad
loss

3.294508457183838
3.2425308227539062
3.2026824951171875
3.2603511810302734
3.1914124488830566
3.2449638843536377
3.0821692943573
3.14800763130188
3.085305690765381
3.0887529850006104
2.8433139324188232
3.087385416030884
2.9587788581848145
2.988227605819702
3.0135598182678223
2.862867593765259
3.041266441345215
2.827094316482544
2.91898250579834
2.9162707328796387
3.140671491622925
2.7561399936676025
3.1472373008728027
2.982574224472046
2.7736563682556152
3.1247506141662598
2.9332525730133057
3.321528196334839
2.847773313522339
3.003903865814209
2.7436110973358154
2.7703685760498047
2.92281174659729
2.961428642272949
3.215855598449707
2.8277971744537354
2.8310439586639404
3.0136635303497314
2.833313226699829
2.8912243843078613
2.835700273513794
2.653529405593872
2.602212905883789
2.493516445159912
2.8296761512756348
2.6190505027770996
3.013400077819824
2.4723832607269287
2.827338695526123
2.8813352584838867
2.68768310546875
2.667628288269043
3.0420050621032715
2.8820483684539795
3.12599

tensor(2.0719, grad_fn=<NllLossBackward0>)

In [173]:
mean_h1 = torch.zeros((1,100)).float()
std_h2 = torch.zeros((1,100)).float()

with torch.no_grad():
    input = C[ix]

    h = input.view(-1,emb_len*cntx_len)@ W1 + b1
    h = torch.tanh(h)
    print(h)
    mean_h1 += h.mean(0,keepdim = True)
    std_h2 += h.std(0, keepdim = True)


tensor([[-0.7746, -0.4600,  0.2109,  ..., -0.8207,  0.3697, -0.9853],
        [-0.9397, -0.6851,  0.0166,  ..., -0.9961, -0.6844,  0.0879],
        [-0.4616, -0.7848,  0.9272,  ..., -0.3350,  0.6759, -0.9803],
        ...,
        [-0.4169,  0.0846, -0.6135,  ..., -0.7489, -0.8987, -0.0598],
        [-0.8668, -0.3599, -0.8109,  ..., -0.7074, -0.9980, -0.2470],
        [-0.3581,  0.8853, -0.6232,  ..., -0.2117, -0.9014, -0.4992]])


In [174]:
running_std,std_h2

(tensor([[0.3408, 0.6784, 0.6353, 0.4353, 0.4302, 0.3459, 0.4221, 0.3388, 0.6054,
          0.5484, 0.5061, 0.4973, 0.5993, 0.6659, 0.4745, 0.5647, 0.6207, 0.4571,
          0.5309, 0.4857, 0.8209, 0.3973, 0.2438, 0.2841, 0.4371, 0.6122, 0.5597,
          0.3897, 0.4612, 0.4399, 0.5398, 0.4819, 0.5167, 0.4780, 0.5958, 0.6546,
          0.5998, 0.5582, 0.3088, 0.7708, 0.5058, 0.2907, 0.4660, 0.4625, 0.5050,
          0.3745, 0.3823, 0.6484, 0.4679, 0.6944, 0.5281, 0.6247, 0.5344, 0.2644,
          0.5383, 0.3928, 0.3103, 0.3031, 0.4560, 0.4663, 0.6405, 0.5895, 0.3083,
          0.4119, 0.6439, 0.4777, 0.6335, 0.4461, 0.7613, 0.4638, 0.6810, 0.5704,
          0.4216, 0.4498, 0.4928, 0.5375, 0.5737, 0.5329, 0.2798, 0.6203, 0.6086,
          0.5723, 0.6476, 0.5200, 0.2684, 0.4369, 0.6245, 0.1712, 0.2538, 0.6540,
          0.5157, 0.5380, 0.5309, 0.4531, 0.5233, 0.3489, 0.5522, 0.6178, 0.6160,
          0.4791]]),
 tensor([[0.3293, 0.6689, 0.6301, 0.4572, 0.4210, 0.3405, 0.4213, 0.3436, 0.6

In [94]:
for i in range(10):
    out  = ""
    cntxt = [0] * cntx_len
    with torch.no_grad():
        while True:
            input = C[cntxt]
            h = input.view(-1,emb_len*cntx_len)@ W1 + b1
            # print(h)
            logits = h@W2 + b2

            prob = torch.softmax(logits, dim = 1)

            ch_int = torch.multinomial(prob, num_samples = 1,).item()
            out += itos [ch_int]

            cntxt = cntxt[1:] + [ch_int]
            if ch_int == 0:
                break

        print(out)

dim.
kori.
jori.
pusth.
jani.
parey.
beyn.
vora.
bery.
alya.


In [233]:
input = torch.randn((500,10), dtype=torch.float32)
out = torch.tanh(input)

In [234]:
input.std(dim = 0,keepdim=True)

tensor([[1.0205, 0.9496, 1.0091, 0.9768, 0.9781, 1.0317, 0.9839, 1.0100, 0.9548,
         1.0128]])

In [235]:
out.std(dim = 0 ,keepdim=True)

tensor([[0.6385, 0.5998, 0.6139, 0.6190, 0.6293, 0.6272, 0.6145, 0.6351, 0.6179,
         0.6315]])

In [236]:
out

tensor([[ 0.6977, -0.5661, -0.8247,  ..., -0.4644,  0.7061, -0.4721],
        [-0.8357, -0.9172, -0.4342,  ..., -0.4698, -0.8648,  0.8850],
        [-0.9540,  0.9941, -0.4373,  ..., -0.5580, -0.8709, -0.5818],
        ...,
        [ 0.5995, -0.2308, -0.7539,  ...,  0.0013,  0.1129, -0.5778],
        [-0.8042, -0.4804, -0.3652,  ..., -0.3577,  0.3221,  0.5688],
        [ 0.7654, -0.1273, -0.4399,  ..., -0.1711, -0.0386,  0.5846]])